# Simulated data generator (library)

Replays games in Gazebo (teleported pieces), randomizes lighting and camera placement per
session, grabs `/camera/image` after each ply and writes labelled square crops +
`manifest.csv` in exactly the format of `gambit_vision/docs/contracts.md`, so the ResNet-18
can be trained **sim only**, **real only** and **mixed** (README "Updated plan (11/4/2026)").
Sim sessions are named `sim_*` so splits by `session_id` keep sim and real apart.

It also holds the camera's **true intrinsics** (from the SDF) and its exact projection, used
by `run_sim_calibration.ipynb` to grade a simulated ChArUco calibration, and to get exact
ground-truth board corners for testing the homography on simulated frames.

Config: `configs/data_generator.yaml`.

**TODO checklist**
- [x] true intrinsics K from the camera SDF, exact world -> pixel projection
- [x] compare an estimated K against the true one
- [x] session naming, lighting / camera-jitter sampling, quaternion helper
- [x] manifest rows (64 per frame, contracts.md columns) and CSV writing
- [ ] `apply_lighting` / `apply_camera_pose` backends (gz service light_config / set_pose)
- [ ] `grab_frame` from `/camera/image` (rclpy + cv_bridge, or `gz topic`)
- [ ] `extract_square_crops`: reuse gambit_vision's homography + crop code (M2), do not duplicate
- [ ] run `generate_session` for real and check the crops by eye

In [ ]:
%run ./teleport.ipynb

In [ ]:
import csv
import math
from datetime import date

import numpy as np

DATA_CFG_PATH = REPO / "configs/data_generator.yaml"
with open(DATA_CFG_PATH) as f:
    DATA_CFG = yaml.safe_load(f)

## True intrinsics from the SDF

Gazebo renders an ideal pinhole camera (no distortion, square pixels). With horizontal field of
view `hfov` and image width `w`:

    fx = fy = (w / 2) / tan(hfov / 2),   cx = w / 2,   cy = h / 2

For hfov = 0.95 rad and 640 x 480: fx ~ 622.0 px. (Check against `/camera/camera_info` once;
some Gazebo versions use (w - 1) / 2 for cx.) A simulated calibration should recover these.

In [ ]:
def camera_params_from_sdf(path=CAMERA_SDF):
    root = ET.parse(path).getroot()
    cam = root.find(".//sensor/camera")
    return {"hfov": float(cam.findtext("horizontal_fov")),
            "width": int(cam.findtext("image/width")),
            "height": int(cam.findtext("image/height"))}

def true_intrinsics(params=None):
    """3x3 K of the simulated camera (distortion is zero)."""
    p = params or camera_params_from_sdf()
    w, h = p["width"], p["height"]
    fx = (w / 2) / math.tan(p["hfov"] / 2)
    return np.array([[fx, 0, w / 2], [0, fx, h / 2], [0, 0, 1.0]])

def compare_intrinsics(K_est, K_true=None, dist=None):
    """Errors of an estimated calibration vs the truth: relative focal error, principal point (px)."""
    K_true = true_intrinsics() if K_true is None else K_true
    out = {"fx_rel_err": (K_est[0, 0] - K_true[0, 0]) / K_true[0, 0],
           "fy_rel_err": (K_est[1, 1] - K_true[1, 1]) / K_true[1, 1],
           "cx_err_px": K_est[0, 2] - K_true[0, 2],
           "cy_err_px": K_est[1, 2] - K_true[1, 2]}
    if dist is not None:
        out["dist_norm"] = float(np.linalg.norm(np.ravel(dist)))   # truth is 0
    return out

## Exact projection (ground truth)

SDF pose `x y z roll pitch yaw` -> rotation R = Rz(yaw) Ry(pitch) Rx(roll). The camera link looks
along +x (y left, z up); the optical frame (OpenCV) is X right = -y_link, Y down = -z_link,
Z forward = x_link. A world point p maps to p_link = R^T (p - t), then u = fx X/Z + cx, v = fy Y/Z + cy.

In [ ]:
def rpy_to_matrix(roll, pitch, yaw):
    cr, sr, cp, sp, cy, sy = math.cos(roll), math.sin(roll), math.cos(pitch), math.sin(pitch), math.cos(yaw), math.sin(yaw)
    Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])
    Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])
    Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])
    return Rz @ Ry @ Rx

def rpy_to_quat(roll, pitch, yaw):
    """(x, y, z, w) quaternion for SDF/ROS roll-pitch-yaw."""
    cr, sr = math.cos(roll / 2), math.sin(roll / 2)
    cp, sp = math.cos(pitch / 2), math.sin(pitch / 2)
    cy, sy = math.cos(yaw / 2), math.sin(yaw / 2)
    return (sr * cp * cy - cr * sp * sy, cr * sp * cy + sr * cp * sy,
            cr * cp * sy - sr * sp * cy, cr * cp * cy + sr * sp * sy)

LINK_TO_OPTICAL = np.array([[0, -1, 0], [0, 0, -1], [1, 0, 0]], dtype=float)

def project_points(pts_world, cam_pose=None, K=None):
    """(N, 3) world points -> (N, 2) pixels for the camera at `cam_pose` (x, y, z, r, p, y)."""
    cam_pose = camera_pose() if cam_pose is None else cam_pose
    K = true_intrinsics() if K is None else K
    t, R = np.array(cam_pose[:3]), rpy_to_matrix(*cam_pose[3:])
    p_opt = (LINK_TO_OPTICAL @ (R.T @ (np.asarray(pts_world, float) - t).T)).T
    if np.any(p_opt[:, 2] <= 0):
        raise ValueError("point behind the camera")
    uv = (K @ (p_opt / p_opt[:, 2:3]).T).T
    return uv[:, :2]

def board_corner_pixels(cam_pose=None, cfg=BOARD):
    """Pixel positions of the playing-area corners (outer corners of a1, h1, h8, a8)."""
    s, z = cfg["board"]["square"] / 2, cfg["board"]["thickness"]
    pts = []
    for sq, (dx, dy) in [("a1", (-s, s)), ("h1", (-s, -s)), ("h8", (s, -s)), ("a8", (s, s))]:
        x, y, _ = square_center(sq, cfg)
        pts.append((x + dx, y + dy, z))
    return dict(zip(["a1", "h1", "h8", "a8"], project_points(pts, cam_pose)))

## Session randomization

One session = one lighting setup + one camera placement (contracts.md: splits are by session).
Sampling is real; *applying* it to Gazebo is a TODO backend. Lights are changed with
`gz service -s /world/gambit_chess/light_config --reqtype gz.msgs.Light` (UserCommands plugin,
already in the world). The camera is moved with set_pose on the `board_camera` model.

In [ ]:
def session_id(index, lighting, fmt=None):
    fmt = fmt or DATA_CFG["sessions"]["name_format"]
    sid = fmt.format(index=index, lighting=lighting, date=date.today().strftime("%Y%m%d"))
    assert sid.startswith("sim_"), "sim session ids must start with sim_"
    return sid

def color_temp_to_rgb(kelvin):
    """Approximate blackbody color (Tanner Helland's fit), channels in 0..1."""
    t = kelvin / 100.0
    r = 255 if t <= 66 else 329.698727446 * (t - 60) ** -0.1332047592
    g = 99.4708025861 * math.log(t) - 161.1195681661 if t <= 66 else 288.1221695283 * (t - 60) ** -0.0755148492
    b = 255 if t >= 66 else (0 if t <= 19 else 138.5177312231 * math.log(t - 10) - 305.0447927307)
    return tuple(min(max(c, 0), 255) / 255 for c in (r, g, b))

def _rotate_dir(d, max_deg, rng):
    d = np.asarray(d, float) / np.linalg.norm(d)
    axis = rng.normal(size=3); axis -= axis.dot(d) * d; axis /= np.linalg.norm(axis)
    a = math.radians(rng.uniform(0, max_deg))
    return d * math.cos(a) + np.cross(axis, d) * math.sin(a)

def sample_lighting(rng, cfg=DATA_CFG):
    L = cfg["lighting"]
    rgb = np.array(color_temp_to_rgb(rng.uniform(*L["sun_color_temp_k"])))
    return {"preset": str(rng.choice(L["presets"])),
            "diffuse": tuple(float(c) for c in 0.85 * rng.uniform(*L["sun_intensity"]) * rgb),
            "direction": tuple(float(c) for c in _rotate_dir((-0.3, -0.2, -1.0), L["sun_direction_jitter_deg"], rng)),
            "ambient": float(rng.uniform(*L["ambient"])),
            "cast_shadows": bool(rng.choice(L["shadows"]))}

def light_config_req(light, name="sun"):
    """gz.msgs.Light request text for /world/<world>/light_config."""
    (r, g, b), (dx, dy, dz) = light["diffuse"], light["direction"]
    return (f'name: "{name}", type: DIRECTIONAL, diffuse: {{r: {r:.3f}, g: {g:.3f}, b: {b:.3f}, a: 1}}, '
            f'direction: {{x: {dx:.4f}, y: {dy:.4f}, z: {dz:.4f}}}, '
            f'cast_shadows: {str(light["cast_shadows"]).lower()}')

def light_config_command(light, world=WORLD_NAME):
    return ["gz", "service", "-s", f"/world/{world}/light_config", "--reqtype", "gz.msgs.Light",
            "--reptype", "gz.msgs.Boolean", "--timeout", "2000", "--req", light_config_req(light)]

def sample_camera_pose(rng, cfg=DATA_CFG, board=BOARD, per_frame=False):
    """Jittered (x, y, z, roll, pitch, yaw) for board_camera, still looking near the board center."""
    J = cfg["camera_jitter"]
    pos_j = J["per_frame_position_m"] if per_frame else J["position_m"]
    pos = np.array(board["camera"]["position"]) + rng.uniform(-pos_j, pos_j, 3)
    look = np.array(board["camera"]["look_at"]) + (0 if per_frame else rng.uniform(-J["look_at_m"], J["look_at_m"], 3))
    jittered = {"camera": {"position": pos.tolist(), "look_at": look.tolist()}}
    _, pitch, yaw = camera_rpy(jittered)
    roll = 0.0 if per_frame else math.radians(rng.uniform(-J["roll_deg"], J["roll_deg"]))
    return (*pos.tolist(), roll, pitch, yaw)

def camera_set_pose(cam_pose):
    return SetPose("board_camera", tuple(cam_pose[:3]), rpy_to_quat(*cam_pose[3:]))

def apply_lighting(light, backend):
    """TODO: send light_config_command(light) (gz CLI) or the ROS equivalent; also set the scene
    ambient (no runtime service for scene ambient in Harmonic: either bake it into a world variant
    per session, or approximate with a second, non-shadow-casting light)."""
    raise NotImplementedError("TODO: apply_lighting")

def apply_camera_pose(cam_pose, backend):
    """TODO: backend.send(camera_set_pose(cam_pose)), then wait a frame so the next image uses it.
    Check that set_pose moves a static model's sensor in your Gazebo version."""
    raise NotImplementedError("TODO: apply_camera_pose")

## Capture (TODO)

`grab_frame` must return a fresh (H, W, 3) uint8 RGB image rendered **after** the last set_pose
(wait `capture.settle_s`, then drop one frame). `extract_square_crops` must produce the crops the
same way as for real frames: ArUco detection -> homography -> warp -> crop (owned by gambit_vision
M2; `%run ../../gambit_vision/notebooks/<capture/homography notebook>` and call it). Do not use
the ground-truth corners for crops of the training data, or sim results will be too optimistic;
use `board_corner_pixels()` only to measure the homography error.

In [ ]:
def grab_frame(timeout_s=2.0):
    """TODO: one RGB frame from /camera/image.
    Option A (ROS): rclpy subscriber + cv_bridge imgmsg_to_cv2(msg, 'rgb8') (see ros_playground).
    Option B (no ROS): `gz topic -e -n 1 -t /camera/image` and parse the gz.msgs.Image bytes."""
    raise NotImplementedError("TODO: grab_frame")

def extract_square_crops(frame):
    """TODO: frame -> (warped board image, {square_name: crop (200, 100, 3)}) via gambit_vision M2."""
    raise NotImplementedError("TODO: extract_square_crops")

## Ground-truth labels -> manifest rows

64 rows per frame in square-index order (a1 = 0 ... h8 = 63), label = python-chess symbol
(`P N B R Q K p n b r q k`) or `empty`, matching `labels.CLASSES`. `path` is relative to the
session directory.

In [ ]:
MANIFEST_COLUMNS = ["session_id", "game_id", "move_idx", "frame_id", "square", "label", "path", "fen"]
CLASSES = ["empty", "P", "N", "B", "R", "Q", "K", "p", "n", "b", "r", "q", "k"]   # = labels.CLASSES

def frame_id(counter):
    return f"{counter:06d}"

def manifest_rows(session, game_id, move_idx, fid, board, ext=".png"):
    rows = []
    for sq in chess.SQUARES:
        p = board.piece_at(sq)
        name = chess.square_name(sq)
        rows.append({"session_id": session, "game_id": game_id, "move_idx": move_idx,
                     "frame_id": fid, "square": name, "label": p.symbol() if p else "empty",
                     "path": f"squares/{fid}_{name}{ext}", "fen": board.board_fen()})
    return rows

def write_manifest(rows, session_dir, append=True):
    """Write / append rows to <session_dir>/manifest.csv (header only when the file is new)."""
    session_dir = Path(session_dir)
    session_dir.mkdir(parents=True, exist_ok=True)
    path = session_dir / "manifest.csv"
    new = not path.exists() or not append
    with open(path, "a" if append else "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=MANIFEST_COLUMNS)
        if new:
            w.writeheader()
        w.writerows(rows)
    return path

def save_crops(session_dir, fid, warped, crops):
    """Write frames/<fid>.png and squares/<fid>_<sq>.png (crops are RGB arrays)."""
    import cv2
    session_dir = Path(session_dir)
    (session_dir / "squares").mkdir(parents=True, exist_ok=True)
    if warped is not None and DATA_CFG["capture"]["save_full_frames"]:
        (session_dir / "frames").mkdir(exist_ok=True)
        cv2.imwrite(str(session_dir / "frames" / f"{fid}.png"), cv2.cvtColor(warped, cv2.COLOR_RGB2BGR))
    for sq, img in crops.items():
        cv2.imwrite(str(session_dir / "squares" / f"{fid}_{sq}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

## One session (orchestration)

The loop is plain plumbing over the pieces above; it fails loudly at the first TODO.

In [ ]:
def generate_session(index, games, backend, out_root=None, cfg=DATA_CFG, seed=0):
    """Replay `games` (list of (game_id, pgn_text)) in one randomized session; returns session dir."""
    rng = np.random.default_rng(seed + index)
    light = sample_lighting(rng, cfg)
    sid = session_id(index, light["preset"], cfg["sessions"]["name_format"])
    out = Path(out_root or REPO / cfg["output"]["root"]) / sid
    apply_lighting(light, backend)
    cam = sample_camera_pose(rng, cfg)
    apply_camera_pose(cam, backend)
    counter = 0
    for game_id, pgn in games:
        for ply in pgn_replay(pgn)[: cfg["games"]["max_plies"] + 1]:
            backend.send_all(ply.requests)
            if ply.idx % cfg["games"]["every_nth_ply"]:
                continue
            board = chess.Board(ply.fen)
            for _ in range(cfg["capture"]["frames_per_ply"]):
                apply_camera_pose(sample_camera_pose(rng, cfg, per_frame=True), backend)
                warped, crops = extract_square_crops(grab_frame())
                fid = frame_id(counter); counter += 1
                save_crops(out, fid, warped, crops)
                write_manifest(manifest_rows(sid, game_id, ply.idx, fid, board), out)
    return out

## Self-check

In [ ]:
import tempfile

K = true_intrinsics()
assert abs(K[0, 0] - 320 / math.tan(0.475)) < 1e-9 and abs(K[0, 0] - 622.0) < 0.5
assert K[0, 2] == 320 and K[1, 2] == 240
err = compare_intrinsics(K * np.array([[1.01, 1, 1], [1, 1.01, 1], [1, 1, 1]]))
assert abs(err["fx_rel_err"] - 0.01) < 1e-9 and err["cx_err_px"] == 0

# the camera looks at the board center -> it projects to the image center
uv = project_points([BOARD["camera"]["look_at"]])[0]
assert np.allclose(uv, [320, 240], atol=1e-6), uv
corners = board_corner_pixels()
assert all(0 < u < 640 and 0 < v < 480 for u, v in corners.values())
# The camera stands beyond rank 8 looking back toward the arm: rank 8 is nearer, so it appears
# LOWER in the image (larger v); the camera's right is world +y, so the a-file is on the RIGHT.
assert corners["a8"][1] > corners["a1"][1] and corners["a1"][0] > corners["h1"][0]

q = rpy_to_quat(0.3, -0.2, 1.1)
w, x, y, z = q[3], q[0], q[1], q[2]
Rq = np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
               [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
               [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])
assert np.allclose(Rq, rpy_to_matrix(0.3, -0.2, 1.1))

rng = np.random.default_rng(0)
light = sample_lighting(rng)
assert light["preset"] in DATA_CFG["lighting"]["presets"] and 'type: DIRECTIONAL' in light_config_req(light)
cam = sample_camera_pose(rng)
assert np.allclose(cam[:3], BOARD["camera"]["position"], atol=DATA_CFG["camera_jitter"]["position_m"] + 1e-9)
assert camera_set_pose(cam).name == "board_camera"
assert session_id(3, "dim") == "sim_03_dim"

b = chess.Board(); b.push_uci("e2e4")
rows = manifest_rows("sim_00_test", "g0001", 1, frame_id(7), b)
assert len(rows) == 64 and list(rows[0]) == MANIFEST_COLUMNS
lab = {r["square"]: r["label"] for r in rows}
assert lab["e4"] == "P" and lab["e2"] == "empty" and lab["e8"] == "k" and set(lab.values()) <= set(CLASSES)
assert rows[28]["path"] == "squares/000007_e4.png"
with tempfile.TemporaryDirectory() as d:
    write_manifest(rows, d); p = write_manifest(rows, d)
    import pandas as pd
    df = pd.read_csv(p)
    assert len(df) == 128 and list(df.columns) == MANIFEST_COLUMNS
print("data_generator self-check OK, fx =", round(K[0, 0], 2))